# Experiment 3.0.1 — Single-$\tau_{syn}$ × objective comparison

## Question
At a fixed 3-layer feature SNN, which homogeneous synaptic time scale is best, and does the answer depend on the training objective?

Backbone: `30 -> 128 -> 128 -> 64`, no recurrence, no bias in SNN linear layers. Within one run, L1/L2/L3 all use the same `shift_syn`. Sweep shifts `[2,3,4,5,6,7]`.

Objectives:
- `timestep_ce`: shared `64 -> 12` classifier at each valid timestep; CE averaged over valid timesteps.
- `relative10_sequence_ce`: valid gesture split into 10 relative bins; count L3 spikes, flatten `10 x 64`, one whole-segment CE.
- `fixed250_sequence_ce`: fixed 250 ms = 16 samples at 64 Hz; count valid L3 spikes per bin, flatten, one whole-segment CE. SNN state is not reset at bin boundaries.

Grid: `6 shifts x 3 objectives x 3 seeds = 54 trainings`, seeds `(11,23,101)`, fixed split seed `12345`.

Evaluation:
1. native test balanced accuracy (primary classification metric), macro-F1, accuracy;
2. common frozen representation probe: for every trained model use the same valid-masked fixed-250-ms L3 representation + `StandardScaler + LogisticRegression` and report test BA;
3. validation BA / best epoch, train-test BA gap, L1/L2/L3 firing rates, zero-input firing rates.

**Execution:** `snn/unity_cpu_runtime.py` detects Unity/Slurm before NumPy/PyTorch import. On Unity it uses CPU and maps CPUs already allocated to the Jupyter kernel to independent training processes. A running notebook cannot retroactively acquire additional Slurm CPUs, so launch the kernel inside the desired allocation. This notebook intentionally keeps the existing timestep-wise SNN forward; it does not apply the separate layer-wise Linear vectorization optimization.

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start=None):
    start=(start or Path.cwd()).resolve()
    for p in (start,*start.parents):
        if (p/'snn').is_dir() and (p/'notebooks').is_dir(): return p
    raise FileNotFoundError('writingRing repo root not found')

REPO_ROOT=find_repo_root()
if str(REPO_ROOT) not in sys.path: sys.path.insert(0,str(REPO_ROOT))

# Run before importing NumPy/PyTorch.
from snn.unity_cpu_runtime import configure_cpu_runtime
RUNTIME=configure_cpu_runtime(total_runs=54,threads_per_run=1,prefer_cpu_on_unity=True,require_slurm_on_unity=True)
RUNTIME.print_summary(total_runs=54)

## 1. Imports and configuration

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
RUNTIME.apply_torch_thread_limits(torch)

from scripts.experiment_3_0_1_single_tau_objectives import (
    EXPERIMENT_ID, EXPECTED_RUNS, SHIFTS, OBJECTIVES, SEEDS, WIDTHS,
    Config, prepare_data, tau_table, run_sweep, aggregate,
)

DEVICE=torch.device('cpu') if RUNTIME.force_cpu else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_PARALLEL=RUNTIME.parallel_runs if DEVICE.type=='cpu' else 1
RESULTS_DIR=REPO_ROOT/'notebooks/artifacts'/EXPERIMENT_ID
CONFIG=Config(REPO_ROOT,RESULTS_DIR,device=DEVICE.type,epochs=100,batch_size=128,resume=True,threads=1)
print('Experiment:',EXPERIMENT_ID)
print('Device:',DEVICE)
print('Shifts:',SHIFTS)
print('Objectives:',OBJECTIVES)
print('Seeds:',SEEDS)
print('Widths:',WIDTHS)
print('Expected runs:',EXPECTED_RUNS)
print('Parallel runs:',N_PARALLEL)
print('DataLoader workers/run: 0')

## 2. Load and validate data

In [ ]:
DATA=prepare_data(REPO_ROOT)
print('sampling rate:',DATA.fs)
print('padded length:',DATA.T)
print('250 ms bin:',DATA.bin_steps,'samples; bins:',DATA.n_bins)
print('labels:',DATA.labels)
print('train/val/test:',DATA.Xtr.shape,DATA.Xva.shape,DATA.Xte.shape)
print('split:',DATA.split)
display(tau_table(DATA.fs))

## 3. Run / resume 54 trainings

In [ ]:
RESULTS=run_sweep(DATA,CONFIG,n_jobs=N_PARALLEL)
print('completed:',len(RESULTS),'/',EXPECTED_RUNS)
display(RESULTS)

## 4. Aggregate mean ± SD over 3 seeds

In [ ]:
SUMMARY=aggregate(RESULTS)
SUMMARY.to_csv(RESULTS_DIR/'experiment_3_0_1_summary.csv',index=False)
display(SUMMARY)

## 5. Native test balanced accuracy vs shift

In [ ]:
fig,ax=plt.subplots(figsize=(8,5))
for obj in OBJECTIVES:
    p=SUMMARY[SUMMARY.objective==obj].sort_values('shift')
    ax.errorbar(p.shift,p.mean_test_balanced_accuracy,yerr=p.sd_test_balanced_accuracy,marker='o',capsize=3,label=obj)
ax.set(xlabel='homogeneous shift_syn in L1/L2/L3',ylabel='test balanced accuracy',xticks=SHIFTS)
ax.grid(alpha=.25); ax.legend(); plt.show()

## 6. Common fixed-250-ms frozen linear-probe BA

In [ ]:
fig,ax=plt.subplots(figsize=(8,5))
for obj in OBJECTIVES:
    p=SUMMARY[SUMMARY.objective==obj].sort_values('shift')
    ax.errorbar(p.shift,p.mean_probe_test_balanced_accuracy,yerr=p.sd_probe_test_balanced_accuracy,marker='o',capsize=3,label=obj)
ax.set(xlabel='homogeneous shift_syn in L1/L2/L3',ylabel='common fixed250 probe test BA',xticks=SHIFTS)
ax.grid(alpha=.25); ax.legend(); plt.show()

## 7. Validation BA vs epoch

In [ ]:
H=pd.read_csv(RESULTS_DIR/'experiment_3_0_1_history.csv')
M=H.groupby(['objective','shift','epoch'],as_index=False).val_balanced_accuracy.mean()
for obj in OBJECTIVES:
    fig,ax=plt.subplots(figsize=(8,5)); p=M[M.objective==obj]
    for s in SHIFTS:
        q=p[p['shift']==s]; ax.plot(q.epoch,q.val_balanced_accuracy,label=f'shift {s}')
    ax.set(title=obj,xlabel='epoch',ylabel='mean validation BA'); ax.grid(alpha=.25); ax.legend(ncol=2); plt.show()

## 8. Firing-rate diagnostics

In [ ]:
cols=['objective','shift','seed','test_l1_firing_rate','test_l2_firing_rate','test_l3_firing_rate','zero_l1_firing_rate','zero_l2_firing_rate','zero_l3_firing_rate']
display(RESULTS[cols])
for obj in OBJECTIVES:
    fig,ax=plt.subplots(figsize=(8,5)); p=RESULTS[RESULTS.objective==obj].groupby('shift',as_index=False)[['test_l1_firing_rate','test_l2_firing_rate','test_l3_firing_rate']].mean()
    for col,label in [('test_l1_firing_rate','L1'),('test_l2_firing_rate','L2'),('test_l3_firing_rate','L3')]: ax.plot(p.shift,p[col],marker='o',label=label)
    ax.set(title=obj,xlabel='shift_syn',ylabel='valid spikes / neuron / timestep',xticks=SHIFTS); ax.grid(alpha=.25); ax.legend(); plt.show()

## 9. Best conditions

In [ ]:
native=RESULTS.groupby(['objective','shift']).test_balanced_accuracy.agg(['mean','std']).sort_values('mean',ascending=False)
probe=RESULTS.groupby(['objective','shift']).probe_test_balanced_accuracy.agg(['mean','std']).sort_values('mean',ascending=False)
print('Native head ranking'); display(native)
print('Common frozen-probe ranking'); display(probe)

## Interpretation
Do not pick a winner from one seed. Use the 3-seed mean and SD. First ask whether a consistent single-$\tau$ optimum exists. Then ask whether the optimum changes with supervision. Finally compare native-head BA with the common fixed250 frozen probe: agreement supports a representation-level effect; disagreement suggests the training/readout head is contributing materially. Inspect firing rates and train-test gap before carrying a single-$\tau$ baseline into the multi-$\tau$ architecture study.